# Chapter 11: Network Defense and Hardening

> "A firewall does not protect you from threats that originate inside it, threats that go around it,
> or threats that walk through its front door." paraphrased from common security teaching

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Describe the role of firewalls, their types, and how to write effective rules.
2. Explain network segmentation and the DMZ architecture.
3. Describe zero-trust architecture and its practical implementation.
4. Configure basic firewall rule sets and explain the implicit deny principle.
5. Explain VPN types and their security properties.
6. Describe DNS security controls including DNSSEC, DoH, and DNS sinkholing.
7. Explain network access control (NAC) and 802.1X authentication.
8. Describe DDoS categories and mitigation strategies.

## Key Terms

- **Firewall**: a device or software that enforces access control between networks.
- **Stateful inspection**: tracks connection state to validate return traffic.
- **NGFW**: Next-Generation Firewall; adds deep packet inspection, application awareness, and IPS.
- **DMZ**: Demilitarised Zone; a network segment between the internet and the internal network.
- **Implicit deny**: any traffic not explicitly permitted is blocked.
- **ACL**: Access Control List; ordered list of permit/deny rules.
- **Zero trust**: security model requiring verification for every request regardless of network location.
- **NAC**: Network Access Control; enforces endpoint compliance before granting network access.
- **802.1X**: IEEE standard for port-based network access control.
- **Sinkhole**: redirecting malicious domain resolutions to a controlled IP for analysis or blocking.
- **Anycast**: routing technique that sends traffic to the nearest of multiple identical endpoints.

---

- **Physical vs virtual firewall**: dedicated hardware appliance vs software/cloud-delivered filtering.
- **Stateless vs stateful filtering**: judging each packet alone vs tracking connections and auto-allowing replies.
- **Proxy / Tor**: relaying intermediary (forward/reverse) vs onion-routed anonymity network.
- **Kerberos / IAM / OAuth / OIDC / FIDO2-WebAuthn**: ticket auth, identity management, delegated authz, sign-in, passwordless.
- **FAR / FRR / EER**: false acceptance, false rejection, and equal-error rates in biometrics.

## Firewalls

### Firewall Types and Evolution

#### Packet-Filter Firewalls

Packet-filter firewalls evaluate each packet independently against a rule set based on header fields:
source/destination IP, port, and protocol. They are fast and simple but cannot track connection
state, making them vulnerable to IP spoofing and fragmentation attacks. They operate at layer 3-4.

#### Stateful Inspection Firewalls

Stateful firewalls maintain a connection table that tracks TCP sessions and UDP pseudo-sessions.
Return traffic is automatically permitted if it belongs to an established, allowed session. This
prevents certain spoofing attacks and simplifies outbound rule writing. The state table itself is
a resource that can be exhausted by SYN floods.

#### Next-Generation Firewalls

NGFWs add layer-7 inspection: application identification (blocking BitTorrent regardless of port),
user identity-based rules (allow finance team to access financial SaaS), URL filtering, TLS
inspection (decrypt-inspect-re-encrypt), and integrated intrusion prevention. NGFWs are the
standard for enterprise perimeter and internal segmentation today.

### Writing Firewall Rules

#### The Implicit Deny Principle

All enterprise firewall rule sets should end with an explicit `deny all` rule (or rely on an
implicit deny that blocks everything not matched above). Traffic is permitted only by explicit
positive rules. This default-deny posture means new services are blocked until consciously allowed,
rather than exposed until someone notices.

#### Rule Ordering and Specificity

Firewall rules are evaluated top to bottom; the first match wins. More specific rules must precede
more general ones. A common misconfiguration is placing a broad permit rule before a more specific
deny, inadvertently allowing everything that the deny was meant to block.

---
- **NetFlow / IPFIX / full packet capture**: network flow metadata and full-byte capture for visibility and forensics.
- **Honeypot / honeynet / honeytoken**: decoy systems, networks, and data used as high-confidence intrusion tripwires.
- **Network forensics**: capturing and analyzing traffic to reconstruct an incident under chain of custody.


<!--DEF11-->
## Firewall Types and Topologies

The firewall overview above introduced the idea of a policy-enforcing chokepoint; here we make the taxonomy
concrete, because choosing the right *type* and *placement* of firewall is what actually determines which data
is protected. Firewalls come in **five basic types**, in roughly increasing sophistication:

- **Packet-filtering** firewalls decide based on IP addresses, packet type, and port, with no awareness of
  connection state.
- **Circuit-level** firewalls set up and validate TCP connections (typically inside-to-outside), operating at
  the session layer.
- **Stateful-inspection** firewalls track the state of each connection, allowing only packets that belong to a
  legitimate, established flow.
- **Application-level** firewalls (proxies) understand specific application traffic and inspect its content.
- **Multilayer** firewalls combine all of the above.

Equally important is **topology**, how firewalls are arranged relative to the assets they protect. A
**host-based firewall** secures the host itself in software (Windows Firewall, iptables, firewalld) and should
run on *every* operating system, including consumer routers. A **bastion host** is a hardened, dedicated
appliance with minimal code through which traffic passes. A **DMZ (demilitarized zone)** places servers between
**two** bastion hosts so that even if the outer, more-exposed host is compromised, the inner host still
protects the internal network. A **distributed** design uses multiple firewalls (for capacity and disaster
recovery) that may share rules, code, and traffic state with one another. Finally, **application-level / web
application firewalls (WAFs)** protect specific applications by enforcing "normalized" traffic: an e-commerce
card field that should contain exactly sixteen digits is rejected when it contains more or fewer, blunting
injection and abuse, though the inspection overhead can be costly.

```{mermaid}
flowchart LR
    NET[Internet] --> OB[Outer bastion host]
    OB --> DMZ[DMZ servers<br/>web, mail, DNS]
    DMZ --> IB[Inner bastion host]
    IB --> LAN[Internal network]
```

```{admonition} Knowledge Check
:class: tip
1. Which firewall type tracks connection state, and why is that stronger than simple packet filtering?
2. In a DMZ with dual bastion hosts, what is protected if the outer host is compromised?
3. Give one reason a large organization runs multiple (distributed) firewalls.

*Answers:* (1) Stateful-inspection firewalls track connection state, so they admit only packets belonging to a
legitimate established flow rather than judging each packet in isolation. (2) The inner bastion host and the
internal network remain protected because the DMZ isolates the exposed servers. (3) Capacity (no single
firewall can handle all traffic) and disaster recovery/redundancy; they can also share rules and state.
```


<!--PHYSVIRT-->
### Physical and Virtual Firewalls

A firewall's *type* (the rule logic of the previous section) is separate from its *form factor*, and the
distinction matters operationally. A **physical (hardware) firewall** is a dedicated appliance, purpose-built
silicon at a network boundary; it offers high, predictable throughput, a hardened minimal operating system, and
a clear physical chokepoint, which is why data centers and enterprise perimeters still rely on them. A
**virtual firewall** is the same filtering logic delivered as software: a virtual appliance in a hypervisor, a
host-based firewall on an operating system (Windows Firewall, iptables/nftables, firewalld), or a cloud-native
control such as the security groups and network ACLs of Chapter 3. Virtual firewalls trade some raw throughput
for elasticity and reach: they scale with workloads, move with virtual machines and containers, and can enforce
policy *east-west* (between workloads inside the same network) where a single perimeter appliance never sees the
traffic. Modern architectures use both, hardware at the physical edge and virtual firewalls woven through the
virtualized and cloud interior, which is exactly the micro-segmentation and zero-trust posture developed below.


### Stateless and Stateful Packet Filtering

The deepest behavioral split among firewalls is whether they remember connections. A **stateless packet
filter** judges every packet in isolation against its rules (source/destination address, port, protocol, flags)
with no memory of what came before. It is fast and simple but blunt: to allow a reply it must have an explicit
rule for the return traffic, and it cannot tell a legitimate response from a forged packet that merely *looks*
like one. A **stateful firewall** maintains a **connection (state) table** tracking each active flow (the TCP
handshake, the expected sequence numbers, the four-tuple of addresses and ports). Once it admits an outbound
request, it **automatically allows the matching return traffic** without a separate inbound rule, and it drops
packets that do not belong to a known, valid connection. This is more secure and far easier to manage, which is
why stateful inspection is the default for perimeter firewalls and why cloud security groups are stateful while
network ACLs are stateless (Chapter 3).

```{admonition} Warning: attacks on the "always allow the response" behavior
:class: warning
A stateful firewall's convenience, automatically trusting return traffic for an established flow, is also an
attack surface. Several techniques abuse it: **TCP state-table exhaustion** floods the firewall with half-open
or idle connections until the state table fills and it fails open or drops legitimate flows (a denial of
service). **Spoofed or injected "response" packets** crafted to match an expected connection (correct ports and
plausible sequence numbers) can slip through as if they were the awaited reply, the basis of off-path TCP
injection. **ACK-flag and "established"-rule bypasses** exploit permissive rules that allow any packet with the
ACK bit set (assumed to belong to an existing connection), letting a crafted ACK or RST traverse a filter, the
reason Nmap's ACK scan (Chapter 8) can map such rule sets. And **connection-reuse or request-smuggling** tricks
ride inside an already-trusted session. The defenses are strict state validation (sequence-number and flag
checks), connection rate-limiting and SYN-cookie protection, dropping rather than trusting unsolicited ACKs,
and pairing the stateful firewall with deeper inspection (Chapter 12). The lesson generalizes: any control that
*infers* trust from prior state can be fooled by forging the evidence of that state.


## Network Segmentation

### DMZ Architecture

A DMZ (Demilitarised Zone) places internet-facing servers (web, mail, DNS) in a network segment
that is reachable from the internet but cannot initiate connections to the internal network. The
DMZ sits between two firewalls: the outer firewall permits inbound traffic to DMZ services; the
inner firewall permits only specific, authorised traffic from the DMZ to the internal network
(e.g., the web server may query the internal database on port 5432 only). If the web server is
compromised, the attacker cannot reach the internal network without breaching the inner firewall.

### VLAN and Micro-Segmentation

VLANs create logical network segments that limit broadcast domains and create enforcement points.
Micro-segmentation extends this to the workload level: even hosts on the same VLAN have explicit
rules about which port/protocol combinations can communicate. East-west traffic (between internal
workloads) is filtered just as rigorously as north-south traffic (to/from internet). Micro-
segmentation dramatically limits lateral movement after initial compromise.

---

## Zero-Trust Architecture

### The Zero-Trust Principle

Zero trust abandons the assumption that everything inside the network perimeter is safe. Every
access request is evaluated against policy regardless of where it originates: inside the office,
on VPN, or from a cloud workload. The three core principles are:

1. Verify explicitly: authenticate and authorise every request using all available data points
   (identity, device health, location, time, data sensitivity).
2. Use least-privilege access: grant only the permissions needed for the specific request.
3. Assume breach: minimise blast radius through segmentation; assume an attacker is already inside
   and design accordingly.

#### Identity-Centric Access

In a zero-trust model, the network is not the trust boundary; the identity is. A user with strong
MFA and a healthy, compliant device gets access. The same user on an unmanaged device gets reduced
access or is denied. A compromised account, even from inside the office, cannot reach sensitive
resources without the correct device posture and MFA.

---

## DNS Security

### DNSSEC

DNSSEC adds cryptographic signatures to DNS resource records. A DNSSEC-aware resolver validates
the chain of signatures from the root zone down to the answer, detecting tampered or forged records.
DNSSEC prevents cache poisoning but does not encrypt DNS traffic (queries and responses remain
visible).

### DNS over HTTPS and DNS over TLS

DoH and DoT encrypt the DNS channel, preventing eavesdropping on queries. DoH sends DNS queries
inside HTTPS traffic on port 443, making it indistinguishable from normal web traffic. DoT uses
a dedicated port (853) and allows enterprise filtering. Enterprise deployments often force DoT to
a corporate resolver that applies sinkholing and filtering policies.

### DNS Sinkholing

A sinkhole redirects DNS queries for known-malicious domains to a controlled IP address rather
than the actual C2 server. Malware that checks in with a sinkholed domain is identified (the
connecting host is infected) and prevented from reaching its real C2. Threat intelligence feeds
supply the domain blocklists; enterprise DNS resolvers apply them automatically.

---

## VPNs and Remote Access

### IPsec and WireGuard

IPsec provides network-layer encryption and authentication, operating in tunnel mode (encrypting
the entire IP packet, including headers) for site-to-site VPNs or transport mode for host-to-host.
WireGuard is a modern, lean VPN protocol that uses state-of-the-art cryptography (Curve25519,
ChaCha20, Poly1305) with a minimal code base, making it faster to audit and in practice faster than
IPsec or OpenVPN.

### Split Tunnelling and Its Risks

Split tunnelling routes only corporate-destined traffic through the VPN while internet traffic goes
directly to the user's ISP. This reduces latency and VPN gateway load but means the VPN provides no
visibility or control over the user's general internet activity. A user infected with malware while
on split-tunnel VPN may continue to communicate with C2 without the corporate security stack seeing
it.

---



<!--PROXYTOR-->
## Proxies, VPNs, and Tor

VPNs (above) are one of three related tools for controlling and concealing where traffic goes, and a defender
should understand all three because each appears on both sides of the fence. A **proxy** is an intermediary that
makes requests on a client's behalf: a **forward proxy** sits in front of users (for web filtering, caching,
logging, and egress control, the corporate gateway that inspects outbound traffic), while a **reverse proxy**
sits in front of servers (for load balancing, TLS termination, and as the front end of a web application
firewall). A proxy sees and can log or modify the traffic it relays, so it is a control point for defenders and
a collection point for attackers who compromise it. A **VPN** (Section above) builds an encrypted tunnel that
extends a host or site into a remote network, protecting confidentiality over untrusted paths and masking the
client's apparent source address behind the VPN endpoint.

**Tor (The Onion Router)** provides a stronger property, *anonymity*, by routing traffic through a volunteer
circuit of three relays with layered ("onion") encryption, so no single relay knows both the origin and the
destination: the entry guard sees the user but not the destination, the exit node sees the destination but not
the user, and the middle relay sees neither. Tor protects against traffic analysis and censorship and is used
by journalists, activists, and, inevitably, criminals; its exit nodes can read unencrypted exit traffic, so
end-to-end encryption (HTTPS) still matters on top of it. The three tools form a spectrum: a proxy redirects
and inspects, a VPN encrypts and relocates, and Tor anonymizes, and the unlinkability and anonymity properties
of Chapter 2 are exactly what Tor aims to deliver in practice.


## Network Access Control and 802.1X

NAC enforces endpoint health before granting network access. Before a device is permitted onto the
network it must present valid credentials (802.1X) and pass a health check (antivirus current,
OS patches applied, disk encryption enabled). Non-compliant devices are quarantined to a remediation
VLAN with access only to update servers.

### 802.1X Operation

In 802.1X, three components interact: the supplicant (client device), the authenticator (switch or
wireless access point), and the authentication server (RADIUS). The switch blocks all traffic from
a newly connected device except EAP (Extensible Authentication Protocol) frames until the
authentication server validates the device's credentials. Only then does the switch permit normal
traffic.

---

## DDoS and Mitigation

### DDoS Attack Categories

| Category | Mechanism | Example |
|---|---|---|
| Volumetric | Saturate bandwidth | DNS amplification, UDP flood |
| Protocol | Exhaust state tables | SYN flood, fragmentation |
| Application-layer | Exhaust server resources | HTTP flood, Slowloris |

### DDoS Mitigation

Cloud-based scrubbing services (Cloudflare, Akamai, AWS Shield) absorb attack traffic using
anycast routing, which distributes the attack across a global network. Rate limiting at the
network edge drops traffic from sources exceeding defined thresholds. BGP blackholing redirects
attack traffic to null routes. Application-layer DDoS mitigation requires distinguishing legitimate
from bot traffic using CAPTCHA, JavaScript challenges, and browser fingerprinting.

---



<!--AUTHID-->
## Authentication, Identity, and Access

Firewalls and segmentation decide where traffic may go; **authentication** decides *who* may act, and in a
zero-trust world (above) identity becomes the real perimeter. Recall the AAA model of Chapter 1:
**authentication** proves who you are, **authorization** decides what you may do, and **accounting** records
what you did. Authentication draws on three classic factors, something you **know** (a password or PIN),
something you **have** (a token, smart card, or phone), and something you **are** (a biometric), and combining
two or more is **multi-factor authentication (MFA)**, the single most effective control against credential
theft.

Several technologies implement identity at scale, and each answers a different question:

- **Kerberos** is the ticket-based authentication protocol at the heart of Windows Active Directory (and many
  Unix realms). A central Key Distribution Center (KDC) issues a time-limited Ticket-Granting Ticket after
  login, which the client exchanges for service tickets, so a user authenticates *once* and accesses many
  services without resending a password. Its reliance on tickets and time also creates the attacks of later
  chapters (pass-the-ticket, Kerberoasting, and the need for synchronized clocks).
- **IAM (Identity and Access Management)** is the broader framework, on premises and especially in the cloud
  (Chapter 17), for managing identities, roles, and least-privilege permissions, defining *who* (users,
  groups, service principals) can do *what* to *which* resources.
- **OAuth 2.0** is an **authorization**-delegation framework, it lets an app act on your behalf (access your
  calendar) without giving it your password, by issuing scoped access tokens; the related **OpenID Connect**
  layers *authentication* on top so a site can offer "sign in with" a provider. (A common exam trap: OAuth by
  itself is about authorization, not authentication.)
- **FIDO2 / WebAuthn** enables **passwordless**, phishing-resistant authentication using public-key
  cryptography: the device holds a private key and proves possession to the server, which stores only the public
  key, so there is no shared secret to phish, replay, or breach. Passkeys are the consumer face of this
  standard.
- **Passwordless authentication** more broadly replaces the knowledge factor with possession and inherence
  (security keys, device biometrics, magic links, one-time codes), removing the weakest link, the reusable,
  guessable, phishable password.


### Biometrics and the Reality of False Positives and Negatives

**Biometric** authentication, fingerprint, face, iris, or voice, uses an inherence factor that cannot be
forgotten and is hard to share, which makes it convenient and increasingly common as the "something you are"
factor. But biometrics are *probabilistic*, not exact: a captured sample never matches the stored template
bit-for-bit, so the system accepts a match within a tolerance, and that tolerance creates two unavoidable error
types that recur throughout detection (Chapter 12) and machine learning (Chapter 17).

- A **false positive** (false acceptance) admits the wrong person, a security failure; its rate is the **False
  Acceptance Rate (FAR)**.
- A **false negative** (false rejection) denies the legitimate person, a usability failure; its rate is the
  **False Rejection Rate (FRR)**.

Tightening the threshold lowers false acceptances but raises false rejections, and loosening it does the
reverse; the operating point where the two rates are equal is the **Equal Error Rate (EER)**, a common
single-number measure of accuracy. The same trade-off is the precision-versus-recall tension behind every IDS,
antivirus, and anomaly detector in this book, which is why biometrics are best used as *one* factor in MFA
rather than alone, and why systems must also resist presentation ("spoofing") attacks such as a photo, mask, or
lifted fingerprint with liveness detection. The deeper point is that any classifier, biometric, signature, or
model, is defined not by a single accuracy number but by where on the false-positive/false-negative curve it
chooses to sit.

```{admonition} Knowledge Check
:class: tip
1. Why does a stateful firewall not need an explicit inbound rule for the reply to an allowed outbound request,
   and how is that behavior abused?
2. What does OAuth 2.0 actually provide, and what standard adds authentication on top of it?
3. In biometrics, what is the trade-off between the False Acceptance Rate and the False Rejection Rate, and what
   is the Equal Error Rate?

*Answers:* (1) It tracks the connection in its state table and automatically allows matching return traffic;
attackers abuse this with state-table exhaustion, spoofed/injected packets matching an expected flow, and
permissive ACK/"established" rules. (2) OAuth 2.0 provides delegated *authorization* (scoped access tokens
without sharing the password); OpenID Connect adds *authentication*. (3) Tightening the match threshold lowers
false acceptances (FAR) but raises false rejections (FRR), and vice versa; the Equal Error Rate is the point
where FAR equals FRR, used as a single accuracy measure.


<!--CH11MON-->
## Network Monitoring and Visibility

Firewalls and segmentation decide what *should* cross the network; monitoring tells you what *actually* does,
and you cannot defend what you cannot see. Network visibility comes at several resolutions. **Flow records**
(Cisco **NetFlow**, the vendor-neutral **IPFIX**, or sFlow) summarize each conversation, who talked to whom, on
what ports, how much data, for how long, which is cheap to store and ideal for spotting beaconing, exfiltration,
and lateral movement. **Full packet capture** records the actual bytes (the sniffing of Chapter 3) for deep
analysis and evidence, at high storage cost. To collect either, sensors are fed by a **SPAN/mirror port** or a
passive network **TAP** so they see traffic without sitting inline.

On top of this telemetry, **network detection and response (NDR)** tools build a **baseline** of normal
behavior and alert on deviations, the anomaly-detection idea of Chapters 12 and 17 applied to flows. The
defensive payoff is that monitoring closes the loop with the offensive techniques earlier in the book: the
scans of Chapter 8, the command-and-control of Chapter 9, and the data theft of a breach all leave traffic that
visibility can catch, which is why a blue team invests as heavily in seeing the network as in filtering it.


## Deception: Honeypots, Honeynets, and Honeytokens

A powerful complement to monitoring is **deception**, deliberately planting attractive, fake assets whose only
legitimate purpose is to be touched by an attacker, so that any interaction is high-confidence evidence of
malice. A **honeypot** is a decoy system (a fake database or RDP server) that records how it is probed and
attacked; classic taxonomy distinguishes **low-interaction** honeypots (emulated services, safe but shallow)
from **high-interaction** ones (real systems, richer intelligence but riskier), and the historical "Generation
I/II" honeynets that chain several together behind a controlled gateway. A **honeynet** is a whole decoy
network; a **honeytoken** is a decoy *piece of data*, a fake credential, an unused admin account, or a "canary"
file or URL that should never be accessed, so an alert on it signals intrusion (these are tamper-evident
tripwires, Chapter 2). Deception's great virtue is its **near-zero false-positive rate**: nobody has a
legitimate reason to log into the honeypot or open the canary file, so an alert is almost always real, which is
why deception is increasingly woven into enterprise defense and not just research.


## Network Forensics in Defense

When monitoring or deception fires, the network record becomes evidence, and **network forensics**, the capture
and analysis of network traffic to reconstruct an incident, is the bridge from defense to investigation
(Chapter 13). The flow records and packet captures above are not only detection telemetry; properly preserved,
they answer the investigator's questions: which host was patient zero, what command-and-control it contacted,
what data left and when. The discipline imposes requirements beyond ordinary monitoring: captures must be
time-synchronized (NTP), integrity-protected (hashed, Chapter 2's tamper-evidence), and handled under a
documented **chain of custody** so they are admissible. This is also where modern research is most active:
AI-driven triage and attribution increasingly help investigators cope with the volume of network and device
evidence, a theme developed fully in Chapter 13. The practical point for the defender is to design logging and
capture *before* an incident, because evidence not collected at the time usually cannot be recovered later.


## CVE Case Study: When the Firewall Is the Door (CVE-2024-3400)

Nothing illustrates the stakes of network defense better than the security device itself becoming the entry
point. **CVE-2024-3400** is a command-injection vulnerability in the GlobalProtect feature of Palo Alto
Networks **PAN-OS** (the operating system of its firewalls), rated the maximum **CVSS 10.0**. It let an
*unauthenticated, remote* attacker execute arbitrary code with **root** privileges on the firewall, the very
device meant to protect the network. Per public reporting (Palo Alto/Volexity/CISA), it was exploited as a
zero-day from late March 2024 in a campaign tracked as *Operation MidnightEclipse*, with public proof-of-concept
code and reset-surviving persistence appearing soon after disclosure.

```{mermaid}
flowchart LR
    A[Unauthenticated request to GlobalProtect] --> B[Arbitrary file creation]
    B --> C[OS command injection]
    C --> D[Root-level code execution on the firewall]
    D --> E[Pivot into the protected internal network]
```

The defensive lessons map onto this whole chapter. First, **internet-facing security appliances are
high-value targets**, not trusted by default, exactly the assume-breach posture of zero trust above. Second,
**rapid patch management** is decisive: the window between disclosure and mass exploitation is now days
(Chapter 8's GreyNoise data showed scanning often precedes disclosure). Third, **defense in depth and least
privilege** limit the blast radius, so that root on the edge device does not equal root everywhere. Fourth,
**monitoring and network forensics** (above) are what detect the post-exploitation pivot when prevention fails.
A firewall is only a control, not a guarantee, and must itself be defended, monitored, and patched like any
other asset.


## Capstone and Group Project Ideas (Network Defense)

The skills in this chapter lend themselves to hands-on capstone work. The following team projects, drawn from
the full catalog in Appendix H, are especially suited to network defense and can be built and demonstrated
ethically in an authorized lab:

- **Lightweight SIEM**: ingest logs and flow records, correlate events, and raise alerts (Chapter 12).
- **Network protocol analyzer and packet sniffer** (Scapy/libpcap): capture and decode traffic, flag anomalies.
- **Wi-Fi security auditing tool** (authorized hardware only): assess wireless posture (Chapter 16).
- **Zero Trust Architecture design proposal**: a reference design migrating an enterprise to ZTA.
- **Cloud security misconfiguration auditing with Infrastructure-as-Code scanning** (Chapter 17).
- **Digital Forensics and Incident Response (DFIR) playbook** for a ransomware scenario (Chapters 13, 14).

```{admonition} Knowledge Check
:class: tip
1. What does a flow record (NetFlow/IPFIX) capture that a firewall log might not, and why is it cheaper than
   full packet capture?
2. Why does a honeytoken produce so few false positives?
3. What is the central irony and lesson of CVE-2024-3400 for network defenders?

*Answers:* (1) A flow record summarizes each conversation (endpoints, ports, bytes, duration) across all
traffic, ideal for spotting beaconing/exfiltration/lateral movement; it stores metadata rather than full
payloads, so it is far smaller than packet capture. (2) Nobody has a legitimate reason to use a decoy
credential or open a canary file, so any access is almost certainly malicious. (3) The security appliance
itself (the firewall) became the unauthenticated root entry point, so security devices must be treated as
high-value targets, patched fast, monitored, and contained by defense in depth and least privilege.
```


## Why This Matters

Network defense is the layer that contains damage once a host is compromised. A firewall that limits
outbound traffic prevents data exfiltration. A DMZ limits blast radius when a web server is
compromised. A NAC-enforced network prevents an unpatched guest device from spreading malware to
corporate assets. Zero-trust architecture means a compromised credential does not automatically
grant access to every resource. These controls work together to shorten the attacker's kill chain.

---

## News in Focus

Several nation-state intrusion campaigns achieved network-wide compromise specifically because
east-west traffic was unfiltered: once the attacker gained a foothold in one workload, they could
reach any other workload on the same flat network using standard protocols (SMB, RDP, WMI). The
entire subsequent kill chain depended on the absence of micro-segmentation. Post-incident
recommendations universally included implementing east-west filtering and zero-trust principles
for workload communication.

---


In [1]:
# Chapter 11 -- Firewall rule evaluator and network segment model

from dataclasses import dataclass
from typing import Optional

@dataclass
class FirewallRule:
    action: str           # PERMIT or DENY
    src_ip: str           # CIDR or "any"
    dst_ip: str           # CIDR or "any"
    protocol: str         # tcp, udp, icmp, any
    dst_port: Optional[int]  # None for "any"
    description: str

def ip_in_cidr(ip, cidr):
    if cidr == "any":
        return True
    if "/" not in cidr:
        return ip == cidr
    base, prefix = cidr.split("/")
    prefix = int(prefix)
    def to_int(addr):
        parts = list(map(int, addr.split(".")))
        return (parts[0]<<24)|(parts[1]<<16)|(parts[2]<<8)|parts[3]
    mask = (0xFFFFFFFF << (32-prefix)) & 0xFFFFFFFF
    return (to_int(ip) & mask) == (to_int(base) & mask)

def evaluate(rules, pkt):
    for i, r in enumerate(rules):
        if not ip_in_cidr(pkt["src"], r.src_ip):
            continue
        if not ip_in_cidr(pkt["dst"], r.dst_ip):
            continue
        if r.protocol != "any" and r.protocol != pkt["proto"]:
            continue
        if r.dst_port is not None and r.dst_port != pkt["port"]:
            continue
        return r.action, i+1, r.description
    return "DENY", 0, "Implicit deny (no rule matched)"

# DMZ firewall rule set
rules = [
    FirewallRule("PERMIT","0.0.0.0/0","203.0.113.10","tcp",443,"Allow HTTPS to web server"),
    FirewallRule("PERMIT","0.0.0.0/0","203.0.113.10","tcp",80, "Allow HTTP to web server"),
    FirewallRule("PERMIT","203.0.113.10","10.0.0.50","tcp",5432,"Web -> DB on port 5432 only"),
    FirewallRule("DENY",  "203.0.113.10","10.0.0.0/8","any",None,"Block DMZ to internal except DB"),
    FirewallRule("PERMIT","10.0.0.0/8","any","any",None,"Allow internal outbound"),
    FirewallRule("DENY",  "any","any","any",None,"Explicit deny all"),
]

packets = [
    dict(src="1.2.3.4",      dst="203.0.113.10", proto="tcp", port=443, label="Internet -> Web HTTPS"),
    dict(src="203.0.113.10", dst="10.0.0.50",    proto="tcp", port=5432,label="Web -> DB query"),
    dict(src="203.0.113.10", dst="10.0.0.1",     proto="tcp", port=22,  label="Web -> Internal SSH (lateral)"),
    dict(src="10.0.0.5",     dst="8.8.8.8",      proto="udp", port=53,  label="Internal -> Internet DNS"),
    dict(src="5.5.5.5",      dst="10.0.0.1",     proto="tcp", port=3389,label="Internet -> Internal RDP (attack)"),
]

print(f"{'Packet':<40} {'Action':<8} {'Rule#':<7} {'Note'}")
print("-" * 90)
for p in packets:
    action, rnum, desc = evaluate(rules, p)
    print(f"{p['label']:<40} {action:<8} {rnum:<7} {desc}")


Packet                                   Action   Rule#   Note
------------------------------------------------------------------------------------------
Internet -> Web HTTPS                    PERMIT   1       Allow HTTPS to web server
Web -> DB query                          PERMIT   3       Web -> DB on port 5432 only
Web -> Internal SSH (lateral)            DENY     4       Block DMZ to internal except DB
Internal -> Internet DNS                 PERMIT   5       Allow internal outbound
Internet -> Internal RDP (attack)        DENY     6       Explicit deny all


## Review Questions (MCQ)

**Q1.** The implicit deny principle means:
A. All traffic is denied until a user logs in  B. Any traffic not explicitly permitted by a rule is blocked  C. Deny rules take precedence over permit rules  D. All outbound traffic is denied by default

**Q2.** A DMZ is placed between:
A. Two internal switches  B. The internet-facing and internal-facing firewalls  C. The VPN and the corporate network  D. The RADIUS server and the switch

**Q3.** In zero-trust architecture, the primary trust boundary is:
A. The firewall  B. The corporate office network  C. Verified identity with device posture  D. The VPN

**Q4.** DNSSEC protects against:
A. DNS query eavesdropping  B. Cache poisoning by signing resource records  C. DDoS against name servers  D. Subdomain takeover

**Q5.** A DNS sinkhole primarily helps defenders:
A. Speed up DNS resolution  B. Identify infected hosts attempting to reach malicious C2 domains  C. Prevent DNSSEC failures  D. Enforce DoT

**Q6.** Split-tunnel VPN poses a security risk because:
A. It is slower than full-tunnel  B. Internet traffic bypasses corporate visibility and security controls  C. It requires 802.1X  D. It uses insecure protocols

**Q7.** In 802.1X, the switch acts as the:
A. Supplicant  B. Authentication server  C. Authenticator  D. Certificate Authority

**Q8.** A SYN flood attack targets which resource?
A. Disk I/O  B. The server's TCP connection-state table  C. SSL certificate validity  D. DNS cache

**Q9.** Anycast routing is used in DDoS mitigation to:
A. Encrypt attack traffic  B. Distribute attack traffic across a global network of scrubbing nodes  C. Block UDP amplification  D. Rate-limit HTTP requests

**Q10.** Micro-segmentation primarily limits:
A. Internet-facing attack surface  B. East-west (lateral) movement within the network  C. DDoS impact  D. DNS poisoning

*Answers: Q1 B, Q2 B, Q3 C, Q4 B, Q5 B, Q6 B, Q7 C, Q8 B, Q9 B, Q10 B.*

## Lab Assignment

**Part A -- Firewall rule audit**: Using the evaluator above, add three additional test packets that expose a gap or misconfiguration in the existing rule set. For each packet, explain what attack scenario it represents and what rule change would prevent it.

**Part B -- DMZ design**: Draw (or describe in table form) a three-tier DMZ architecture for a company that hosts a public web application, an internal HR system, and a database server. Specify the firewall rules between each zone.

**Part C -- Zero-trust gap analysis**: For an organisation that currently uses a traditional VPN model, identify five specific changes needed to implement zero-trust principles. For each, specify the technology or process change and the risk it addresses.

**Part D -- DNS security audit**: Run `dig +dnssec example.com` and `dig +dnssec google.com`. Check whether DNSSEC is enabled (look for RRSIG records). Then check whether your local resolver enforces DNSSEC validation. Document your findings and any differences between sites.

## References

```{bibliography}
:filter: docname in docnames
```

1. Practical Computer Security (Course 3): lecture on Types of Firewalls and Configurations (packet, circuit, stateful, application, multilayer; host/bastion/DMZ/distributed/WAF).
2. Neuman, B. C., and Ts'o, T. (1994). Kerberos: An Authentication Service for Computer Networks. IEEE Communications.
3. FIDO Alliance / W3C. FIDO2 and Web Authentication (WebAuthn); Hardt, D. (2012). The OAuth 2.0 Authorization Framework, RFC 6749.
4. Palo Alto Networks; Volexity; CISA (2024). CVE-2024-3400, PAN-OS GlobalProtect command injection (Operation MidnightEclipse). https://security.paloaltonetworks.com/CVE-2024-3400
5. CISA (2023). CVE-2023-4966 (Citrix Bleed) guidance. https://www.cisa.gov/



```{index} Firewall, Stateful inspection, NGFW, DMZ, Implicit deny, ACL, Zero trust, NAC, 802.1X, Sinkhole, Anycast, Physical vs virtual firewall, Stateless vs stateful filtering, Proxy / Tor, Kerberos / IAM / OAuth / OIDC / FIDO2-WebAuthn, FAR / FRR / EER
```
